# Check PCP overlap between subset trees

For each grouping dimension (host, geographic, temporal), check whether parent-child pairs (PCPs) overlap between subsets. We look at two types of overlap:
1. **Internal nodes**: parent sequences shared between subsets (parents of branches are internal nodes, so this is the fraction of identical internal nodes)
2. **Branches**: (parent, child) pairs shared between subsets (a branch is uniquely identified by its endpoints, so this is the fraction of identical branches)

**Expectations:**
- Host subsets (human, avian): small overlap
- Temporal subsets (early, late): small overlap
- Geographic subsets (north_america, europe, asia): substantial overlap

In [1]:
import os
import itertools
import yaml
import pandas as pd

In [2]:
# Load config
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

segments = config['segments']
ha_subtypes = config['ha_subtypes']
na_subtypes = config['na_subtypes']

subset_groups = {
    'host': config['host_groups'],
    'geographic': config['geographic_groups'],
    'temporal': config['temporal_groups'],
}


def get_subtypes_for_segment(segment):
    if segment == 'HA':
        return ha_subtypes
    elif segment == 'NA':
        return na_subtypes
    else:
        return ['all']

In [3]:
def load_subset_pcps(subset_type, groups, segment, subtype):
    """Return a dict mapping group label -> PCP DataFrame for a given subset_type.

    For the 'host' dimension we load the single host-stratified PCP file (which
    has a 'host' column) and split it by host. For other dimensions we load one
    file per subset.
    """
    pcps = {}
    if subset_type == 'host':
        f = f'../results/{segment}/{subtype}/host_stratified/parent_child_pairs.csv'
        if not os.path.exists(f):
            print(f'WARNING: missing {f}')
            return pcps
        df = pd.read_csv(f)
        for group in groups:
            sub = df[df['host'] == group]
            if not sub.empty:
                pcps[group] = sub
    else:
        for group in groups:
            f = f'../results/{segment}/{subtype}/{group}/parent_child_pairs.csv'
            if not os.path.exists(f):
                print(f'WARNING: missing {f}')
                continue
            pcps[group] = pd.read_csv(f)
    return pcps


def compute_overlap(df_a, df_b, columns):
    """Compute overlap between two PCP DataFrames on the given columns.

    Returns (n_shared, frac_a, frac_b) where frac_a is the fraction of
    df_a's unique tuples that also appear in df_b, and vice versa.
    """
    set_a = set(df_a[columns].itertuples(index=False, name=None))
    set_b = set(df_b[columns].itertuples(index=False, name=None))
    shared = set_a & set_b
    n_shared = len(shared)
    frac_a = n_shared / len(set_a) if set_a else 0.0
    frac_b = n_shared / len(set_b) if set_b else 0.0
    return n_shared, frac_a, frac_b

In [4]:
# Compute overlap for all subset pairs across all segment/subtype combinations
rows = []
for subset_type, groups in subset_groups.items():
    for segment in segments:
        for subtype in get_subtypes_for_segment(segment):
            pcps = load_subset_pcps(subset_type, groups, segment, subtype)

            # Compare each pair
            for a, b in itertools.combinations(pcps.keys(), 2):
                # Parents of branches are internal nodes, so parent-only overlap
                # quantifies fraction of identical internal nodes. Parent+child
                # uniquely identifies a branch, so that overlap quantifies
                # fraction of identical branches.
                for label, cols in [('internal nodes', ['parent']),
                                    ('branches', ['parent', 'child'])]:
                    n_shared, frac_a, frac_b = compute_overlap(pcps[a], pcps[b], cols)
                    rows.append({
                        'subset_type': subset_type,
                        'segment': segment,
                        'subtype': subtype,
                        'subset_a': a,
                        'subset_b': b,
                        'feature': label,
                        'n_unique_a': len(set(pcps[a][cols].itertuples(index=False, name=None))),
                        'n_unique_b': len(set(pcps[b][cols].itertuples(index=False, name=None))),
                        'n_shared': n_shared,
                        'frac_a_shared': frac_a,
                        'frac_b_shared': frac_b,
                    })

overlap_df = pd.DataFrame(rows)

# Clean up display names
def prettify(s):
    return s.replace('_', ' ').title()

for col in ['subset_a', 'subset_b', 'subset_type']:
    overlap_df[col] = overlap_df[col].map(prettify)

print(f'{len(overlap_df)} comparisons')
overlap_df.head()

216 comparisons


,subset_type,segment,subtype,subset_a,subset_b,feature,n_unique_a,n_unique_b,n_shared,frac_a_shared,frac_b_shared
0,Host,PB2,all,Human,Avian,internal nodes,33768,15797,0,0.0,0.0
1,Host,PB2,all,Human,Avian,branches,122262,33195,0,0.0,0.0
2,Host,PB2,all,Human,Swine,internal nodes,33768,4092,0,0.0,0.0
3,Host,PB2,all,Human,Swine,branches,122262,6812,0,0.0,0.0
4,Host,PB2,all,Avian,Swine,internal nodes,15797,4092,0,0.0,0.0


## Aggregated overlap fractions by subset pair

Fraction shared recomputed from summed counts across all segment/subtype combinations (i.e., sum n_shared / sum n_unique).

In [5]:
import altair as alt

agg = (
    overlap_df
    .groupby(['subset_type', 'subset_a', 'subset_b', 'feature'])[['n_unique_a', 'n_unique_b', 'n_shared']]
    .sum()
    .reset_index()
)
agg['frac_a_shared'] = agg['n_shared'] / agg['n_unique_a']
agg['frac_b_shared'] = agg['n_shared'] / agg['n_unique_b']

# Melt to get one row per direction
agg_melted = agg.melt(
    id_vars=['subset_type', 'subset_a', 'subset_b', 'feature',
             'n_unique_a', 'n_unique_b', 'n_shared'],
    value_vars=['frac_a_shared', 'frac_b_shared'],
    var_name='direction',
    value_name='fraction_shared',
)
agg_melted['label'] = agg_melted.apply(
    lambda r: f"A={r['subset_a']}, B={r['subset_b']}"
    if r['direction'] == 'frac_a_shared'
    else f"A={r['subset_b']}, B={r['subset_a']}",
    axis=1,
)
# For tooltip: show which n_unique applies to this direction
agg_melted['n_unique'] = agg_melted.apply(
    lambda r: r['n_unique_a'] if r['direction'] == 'frac_a_shared' else r['n_unique_b'],
    axis=1,
)

# Shared x-axis domain across all subplots
x_max = agg_melted['fraction_shared'].max()
x_domain = [0, x_max * 1.05]

agg_charts = []
for subset_type in overlap_df['subset_type'].unique():
    sub = agg_melted[agg_melted['subset_type'] == subset_type]

    chart = (
        alt.Chart(sub, title=f'{subset_type} subsets')
        .mark_circle(size=60)
        .encode(
            x=alt.X('fraction_shared:Q', title='Fraction in subset A with identical matches in subset B',
                     scale=alt.Scale(domain=x_domain)),
            y=alt.Y('label:N', title=None, axis=alt.Axis(grid=True)),
            color=alt.Color('feature:N', title='Feature'),
            tooltip=[
                alt.Tooltip('label:N', title='comparison'),
                alt.Tooltip('feature:N'),
                alt.Tooltip('fraction_shared:Q', format='.4f'),
                alt.Tooltip('n_shared:Q', format=','),
                alt.Tooltip('n_unique:Q', format=','),
            ],
        )
        .properties(width=400, height=20 * sub['label'].nunique())
    )
    agg_charts.append(chart)

agg_chart = alt.vconcat(
    *agg_charts,
    title=alt.TitleParams(
        text='Fraction of identical internal nodes or branches between subsets',
        offset=20,
    ),
).resolve_scale(color='shared')

agg_chart.save('../results/figures/subset_pcp_overlap_aggregated.png', ppi=300)
agg_chart

alt.VConcatChart(...)

## Swarm plots of overlap fractions by subset pair

In [6]:
# Melt so frac_a_shared and frac_b_shared become rows
melted = overlap_df.melt(
    id_vars=['subset_type', 'segment', 'subtype', 'subset_a', 'subset_b', 'feature'],
    value_vars=['frac_a_shared', 'frac_b_shared'],
    var_name='direction',
    value_name='fraction_shared',
)
melted['label'] = melted.apply(
    lambda r: f"A={r['subset_a']}, B={r['subset_b']}"
    if r['direction'] == 'frac_a_shared'
    else f"A={r['subset_b']}, B={r['subset_a']}",
    axis=1,
)

charts = []
for subset_type in overlap_df['subset_type'].unique():
    sub = melted[melted['subset_type'] == subset_type]

    chart = (
        alt.Chart(sub, title=f'{subset_type} subsets')
        .mark_circle(size=40, opacity=0.6)
        .encode(
            x=alt.X('fraction_shared:Q', title='Fraction in subset A with identical matches in subset B'),
            y=alt.Y('label:N', title=None, axis=alt.Axis(grid=True)),
            color=alt.Color('feature:N', title='Feature'),
            tooltip=[
                alt.Tooltip('segment:N'),
                alt.Tooltip('subtype:N'),
                alt.Tooltip('feature:N'),
                alt.Tooltip('fraction_shared:Q', format='.4f'),
                alt.Tooltip('label:N', title='comparison'),
            ],
            yOffset='jitter:Q',
        )
        .transform_calculate(jitter='sqrt(-2*log(random()))*cos(2*PI*random())')
        .properties(width=400, height=40 * sub['label'].nunique())
    )
    charts.append(chart)

swarm_chart = alt.vconcat(
    *charts,
    title=alt.TitleParams(
        text='Fraction of identical internal nodes or branches between subsets',
        offset=20,
    ),
).resolve_scale(color='shared')

swarm_chart.save('../results/figures/subset_pcp_overlap_swarm.png', ppi=300)
swarm_chart

alt.VConcatChart(...)

## Full detail per segment/subtype

In [7]:
with pd.option_context('display.max_rows', None):
    display(overlap_df.sort_values(['subset_type', 'feature', 'segment', 'subtype']))

,subset_type,segment,subtype,subset_a,subset_b,feature,n_unique_a,n_unique_b,n_shared,frac_a_shared,frac_b_shared
107,Geographic,HA,H1,North America,Europe,branches,32216,26781,895,0.027781,0.033419
109,Geographic,HA,H1,North America,Asia,branches,32216,18302,645,0.020021,0.035242
111,Geographic,HA,H1,Europe,Asia,branches,26781,18302,782,0.029200,0.042728
113,Geographic,HA,H3,North America,Europe,branches,30285,25064,882,0.029123,0.035190
115,Geographic,HA,H3,North America,Asia,branches,30285,18535,633,0.020901,0.034152
117,Geographic,HA,H3,Europe,Asia,branches,25064,18535,762,0.030402,0.041111
119,Geographic,HA,H5,North America,Europe,branches,8536,4738,4,0.000469,0.000844
121,Geographic,HA,H5,North America,Asia,branches,8536,7284,4,0.000469,0.000549
123,Geographic,HA,H5,Europe,Asia,branches,4738,7284,33,0.006965,0.004530
125,Geographic,HA,H7,North America,Europe,branches,790,371,0,0.000000,0.000000
